# App-17b : Vehicle Routing Problem (VRP) — Twin C# (métaheuristiques from-scratch)

**Twin C# (.NET Interactive)** de [App-17-VRP-Logistics](App-17-VRP-Logistics.ipynb) — marathon **#4956** (parité .NET ⇄ Python), volet **Search/Applications/Hybrid routing**, axe-2 SOTA **#3801** Prong B.

Le notebook Python invoque **OR-Tools `RoutingModel`** (solveur SOTA industriel) pour le Vehicle Routing Problem capacité (CVRP). Ce twin **déroule les métaheuristiques from-scratch** (BCL .NET 9, **0 NuGet, 0 OR-Tools**) que le solveur black-box orchestre en interne : construction gloutonne, recherche locale 2-opt, recuit simulé.

## Objectifs d'apprentissage

1. Modéliser un VRP capacité (dépôt, clients, demandes, matrice de distance, capacité véhicule).
2. Construire des tournées faisables par **heuristiques gloutonnes** (plus proche voisin, insertion au moindre coût).
3. Améliorer par **recherche locale 2-opt** (inversion de segments intra-tournée).
4. Optimiser globalement par **recuit simulé** (acceptation Metropolis de mouvements défavorables).
5. Comparer les compromis **qualité / temps / contrôle** vs le solveur OR-Tools du twin Python.


## Introduction

### Définition

Le **Vehicle Routing Problem (VRP)** consiste à concevoir des tournées optimales pour une flotte de véhicules desservant un ensemble de clients depuis un dépôt, en minimisant la distance totale parcourue. La variante **capacitaire (CVRP)** ajoute : chaque véhicule a une capacité maximale, chaque client a une demande.

**Données** : un dépôt, $n$ clients (positions 2D, demandes), $k$ véhicules de capacité $Q$.
**Objectif** : minimiser $\sum_{\text{tournées}} \text{distance}$, en visitant chaque client **exactement une fois**, sans dépasser $Q$ par véhicule.

### Complexité

Le VRP est **NP-difficile** (généralise le TSP). Sur $n$ clients et $k$ véhicules, l'espace des partitions faisables croît de manière explosive. En pratique, on utilise :
- **Heuristiques constructives** (glouton, insertion) — rapides, solution de qualité moyenne.
- **Recherche locale** (2-opt, OR-opt) — améliore une solution en explorant le voisinage.
- **Métaheuristiques** (recuit simulé, tabou, algorithmes génétiques) — échappent les optima locaux.
- **Solveurs exacts/SOTA** (OR-Tools `RoutingModel` + Guided Local Search) — qualité industrielle.

### Complémentarité (#3801 Prong B)

| Aspect | Python (twin) | Twin C# (ici) |
|--------|---------------|----------------|
| Solveur SOTA | OR-Tools `RoutingModel` + GLS | **recuit simulé from-scratch** |
| Construction | `greedy_insertion` | **NN + cheapest-insertion** |
| Amélioration | 2-opt | **2-opt intra-route** |
| Métaheuristique | exercice (stub) | **recuit simulé réel** |
| Dépendances | `ortools`, `numpy` | **BCL seule, 0 NuGet** |

Le twin C# rend **visibles** les briques que `RoutingModel` orchestre (construction, voisinage, critère d'acceptation, schéma de refroidissement). OR-Tools reste l'outil SOTA de production (cf twin Python).


## 1. Modélisation — `VRPInstance`

On représente l'instance par le dépôt, la liste des clients (positions 2D), les demandes, la capacité véhicule et le nombre de véhicules. La **matrice de distance euclidienne** est précalculée (index 0 = dépôt, index $i+1$ = client $i$).


In [1]:
// Cellule 1 — Définition de l'instance VRP (persistante cross-cell via le kernel .net-csharp)
using System;
using System.Collections.Generic;
using System.Linq;

public class VRPInstance
{
    public (double X, double Y) Depot { get; }
    public List<(double X, double Y)> Clients { get; }
    public List<int> Demands { get; }
    public int VehicleCapacity { get; }
    public int NumVehicles { get; }
    public double[,] Dist { get; }

    public VRPInstance((double X, double Y) depot, List<(double X, double Y)> clients,
                       List<int> demands, int vehicleCapacity, int numVehicles)
    {
        if (clients.Count != demands.Count)
            throw new ArgumentException("Clients et demands doivent avoir la meme taille.");
        Depot = depot; Clients = clients; Demands = demands;
        VehicleCapacity = vehicleCapacity; NumVehicles = numVehicles;
        Dist = ComputeDistanceMatrix();
    }

    private double[,] ComputeDistanceMatrix()
    {
        // Noeud 0 = depot, noeud i+1 = client i.
        var pts = new List<(double X, double Y)> { Depot };
        pts.AddRange(Clients);
        int n = pts.Count;
        var m = new double[n, n];
        for (int i = 0; i < n; i++)
            for (int j = 0; j < n; j++)
                if (i != j)
                {
                    double dx = pts[i].X - pts[j].X;
                    double dy = pts[i].Y - pts[j].Y;
                    m[i, j] = Math.Sqrt(dx * dx + dy * dy);
                }
        return m;
    }

    public int NumClients => Clients.Count;
}

display("VRPInstance definie. Distance matrix precalculee (index 0 = depot).");


The below script needs to be able to find the current output cell; this is an easy method to get it.

VRPInstance definie. Distance matrix precalculee (index 0 = depot).

### Création de l'instance

On reprend l'instance canonique du twin Python : 15 clients, capacité 60, 6 véhicules. La demande totale (328) dépasse $4 \times 60 = 240$ : **5 véhicules minimum** sont nécessaires, d'où $k=6$ pour marge de manœuvre.


In [2]:
// Cellule 2 — Instance canonique (15 clients, capacite 60, 6 vehicules)
var depot = (50.0, 50.0);
var clients = new List<(double X, double Y)>
{
    (45, 68), (45, 70), (42, 66), (40, 69), (40, 66),
    (38, 58), (38, 65), (35, 66), (35, 69), (25, 85),
    (22, 75), (22, 85), (20, 80), (20, 85), (18, 75)
};
var demands = new List<int> { 10, 15, 18, 15, 20, 25, 20, 18, 22, 30, 35, 25, 20, 30, 25 };

var instance = new VRPInstance(depot, clients, demands, vehicleCapacity: 60, numVehicles: 6);

display($"Depot: {instance.Depot}");
display($"Clients: {instance.NumClients}");
display($"Demande totale: {instance.Demands.Sum()}");
display($"Capacite vehicule: {instance.VehicleCapacity}");
display($"Vehicules: {instance.NumVehicles} (min theorique = ceil(328/60) = 6)");


Depot: (50, 50)

Clients: 15

Demande totale: 328

Capacite vehicule: 60

Vehicules: 6 (min theorique = ceil(328/60) = 6)

## 2. Représentation d'une solution — `VRPSolution`

Une solution = une liste de tournées (chaque tournée = liste d'indices de clients, 0-based). Deux méthodes clés : `TotalDistance()` (coût objectif) et `IsValid()` (tous les clients visités une seule fois + capacité respectée).


In [3]:
// Cellule 3 — Solution VRP + validateur
public class VRPSolution
{
    public List<List<int>> Routes { get; }
    public VRPInstance Instance { get; }

    public VRPSolution(List<List<int>> routes, VRPInstance instance)
    {
        Routes = routes; Instance = instance;
    }

    // Distance d'une seule tournee (depart depot -> clients -> retour depot).
    public double RouteDistance(List<int> route)
    {
        if (route.Count == 0) return 0.0;
        double total = Instance.Dist[0, route[0] + 1];            // depot -> 1er client
        for (int i = 0; i < route.Count - 1; i++)
            total += Instance.Dist[route[i] + 1, route[i + 1] + 1];
        total += Instance.Dist[route[route.Count - 1] + 1, 0];   // dernier -> depot
        return total;
    }

    public double TotalDistance() => Routes.Sum(r => RouteDistance(r));

    public bool IsValid()
    {
        // (a) chaque client visite exactement une fois
        var seen = new HashSet<int>();
        foreach (var route in Routes)
            foreach (var c in route)
                if (!seen.Add(c)) return false;       // doublon
        if (seen.Count != Instance.NumClients) return false;

        // (b) capacite respectee sur chaque tournee
        foreach (var route in Routes)
        {
            int load = route.Sum(c => Instance.Demands[c]);
            if (load > Instance.VehicleCapacity) return false;
        }
        return true;
    }

    public int RouteLoad(List<int> route) => route.Sum(c => Instance.Demands[c]);
}

display("VRPSolution definie. Methods: TotalDistance, IsValid, RouteDistance, RouteLoad.");


VRPSolution definie. Methods: TotalDistance, IsValid, RouteDistance, RouteLoad.

## 3. Heuristique constructive — Plus Proche Voisin (Nearest-Neighbor)

**Principe** : depuis la position courante (dépôt au départ), on va au client non visité le plus proche dont la demande entre dans la capacité restante. Quand plus aucun client ne tient, on retourne au dépôt et on entame une nouvelle tournée.

C'est l'heuristique la plus rapide ($O(n^2)$) mais aussi la plus myope : elle entasse les clients sans vision globale.


In [4]:
// Cellule 4 — Nearest-Neighbor greedy
public VRPSolution NearestNeighbor(VRPInstance inst)
{
    var remaining = new HashSet<int>(Enumerable.Range(0, inst.NumClients));
    var routes = new List<List<int>>();

    while (remaining.Count > 0)
    {
        var route = new List<int>();
        int load = 0;
        int current = 0; // index du noeud courant (0 = depot), dans l'espace matrice (0..n)

        while (true)
        {
            // Chercher le client non visite le plus proche dont la demande tient.
            int bestClient = -1;
            double bestDist = double.PositiveInfinity;
            foreach (var c in remaining)
            {
                if (load + inst.Demands[c] > inst.VehicleCapacity) continue;
                double d = inst.Dist[current, c + 1];
                if (d < bestDist) { bestDist = d; bestClient = c; }
            }
            if (bestClient == -1) break;            // aucun client ne tient : tournee fermee
            route.Add(bestClient);
            load += inst.Demands[bestClient];
            current = bestClient + 1;
            remaining.Remove(bestClient);
        }
        routes.Add(route);
    }
    return new VRPSolution(routes, inst);
}

var nnSolution = NearestNeighbor(instance);
display($"NN — valide: {nnSolution.IsValid()}");
display($"NN — distance totale: {nnSolution.TotalDistance():F2}");
display($"NN — nb tournees: {nnSolution.Routes.Count}");


NN — valide: True

NN — distance totale: 415,63

NN — nb tournees: 6

### Heuristique d'insertion au moindre coût (Cheapest-Insertion)

Variante constructive : on insère chaque client là où l'**augmentation marginale de distance** est minimale. Moins myope que le NN car elle évalue le coût d'insertion dans toutes les tournées existantes. C'est l'heuristique implémentée dans le twin Python (`greedy_insertion`).


In [5]:
// Cellule 5 — Cheapest-Insertion greedy (augmentation marginale minimale)
public VRPSolution CheapestInsertion(VRPInstance inst)
{
    var remaining = new Queue<int>(Enumerable.Range(0, inst.NumClients));
    var routes = new List<List<int>>();
    for (int v = 0; v < inst.NumVehicles; v++) routes.Add(new List<int>());

    while (remaining.Count > 0)
    {
        double bestCost = double.PositiveInfinity;
        int bestRoute = -1, bestClient = -1, bestPos = 0;

        foreach (var client in remaining.ToList())
        {
            for (int r = 0; r < routes.Count; r++)
            {
                var route = routes[r];
                int load = route.Sum(c => inst.Demands[c]) + inst.Demands[client];
                if (load > inst.VehicleCapacity) continue;

                // Essayer chaque position d'insertion dans la tournee.
                for (int pos = 0; pos <= route.Count; pos++)
                {
                    int prev = (pos == 0) ? 0 : route[pos - 1] + 1;     // noeud matrice
                    int next = (pos == route.Count) ? 0 : route[pos] + 1;
                    double cost = inst.Dist[prev, client + 1] + inst.Dist[client + 1, next]
                                - (prev == next ? 0 : inst.Dist[prev, next]);
                    if (cost < bestCost)
                    {
                        bestCost = cost; bestRoute = r; bestClient = client; bestPos = pos;
                    }
                }
            }
        }
        if (bestClient == -1) break;   // plus aucune insertion faisable
        routes[bestRoute].Insert(bestPos, bestClient);
        remaining = new Queue<int>(remaining.Where(c => c != bestClient));
    }
    return new VRPSolution(routes.Where(r => r.Count > 0).ToList(), inst);
}

var greedySolution = CheapestInsertion(instance);
display($"Cheapest-Insertion — valide: {greedySolution.IsValid()}");
display($"Cheapest-Insertion — distance totale: {greedySolution.TotalDistance():F2}");
display($"Cheapest-Insertion — nb tournees: {greedySolution.Routes.Count}");


Cheapest-Insertion — valide: True

Cheapest-Insertion — distance totale: 413,69

Cheapest-Insertion — nb tournees: 6

## 4. Recherche locale — 2-opt intra-route

**Principe** : sur une même tournée, on inverse un segment $[i+1, j]$ et on garde la modification si elle réduit la distance. Le 2-opt élimine les **croisements** d'arcs (sous-optimalité locale typique des heuristiques constructives). On répète jusqu'à ne plus pouvoir améliorer (optimum local).

L'opérateur 2-opt est le cœur des métaheuristiques VRP : il définit le **voisinage** que le recuit simulé explore.


In [6]:
// Cellule 6 — 2-opt local search (intra-route, optimum local)
public List<int> TwoOptRoute(List<int> route, VRPInstance inst)
{
    if (route.Count < 3) return new List<int>(route);
    var best = new List<int>(route);
    bool improved = true;
    while (improved)
    {
        improved = false;
        for (int i = 0; i < best.Count - 1; i++)
        {
            for (int j = i + 1; j < best.Count; j++)
            {
                var candidate = new List<int>(best);
                candidate.Reverse(i + 1, j - i);   // inverse le segment [i+1..j]
                if (RouteDistanceOf(candidate, inst) < RouteDistanceOf(best, inst) - 1e-9)
                {
                    best = candidate;
                    improved = true;
                }
            }
        }
    }
    return best;
}

// Helper statique (evite la dependance cross-cell a une instance de VRPSolution).
private static double RouteDistanceOf(List<int> route, VRPInstance inst)
{
    if (route.Count == 0) return 0.0;
    double total = inst.Dist[0, route[0] + 1];
    for (int i = 0; i < route.Count - 1; i++)
        total += inst.Dist[route[i] + 1, route[i + 1] + 1];
    total += inst.Dist[route[route.Count - 1] + 1, 0];
    return total;
}

public VRPSolution TwoOpt(VRPInstance inst, VRPSolution sol)
{
    var improvedRoutes = sol.Routes.Select(r => TwoOptRoute(r, inst)).ToList();
    return new VRPSolution(improvedRoutes, inst);
}

var twoOptSolution = TwoOpt(instance, greedySolution);
display($"Greedy                  — distance: {greedySolution.TotalDistance():F2}");
display($"Greedy + 2-opt          — distance: {twoOptSolution.TotalDistance():F2}");
display($"Amelioration 2-opt      — {greedySolution.TotalDistance() - twoOptSolution.TotalDistance():F2}");
display($"2-opt valide: {twoOptSolution.IsValid()}");


Greedy                  — distance: 413,69

Greedy + 2-opt          — distance: 413,69

Amelioration 2-opt      — 0,00

2-opt valide: True

## 5. Métaheuristique — Recuit Simulé (Simulated Annealing)

Le 2-opt converge vers un **optimum local** dont il ne peut plus sortir. Le **recuit simulé** (Kirkpatrick 1983) lève ce verrou en acceptant, avec une probabilité décroissante, des mouvements qui **dégradent** la solution — autorisant l'exploration de l'espace au-delà de l'optimum local.

**Schéma** :
1. Solution initiale $s_0$ (ici : greedy + 2-opt).
2. Température initiale $T_0$, schéma de refroidissement géométrique $T \leftarrow \alpha \cdot T$.
3. À chaque itération : tirer un voisin $s'$ (relocate ou 2-opt aléatoire), calculer $\Delta = f(s') - f(s)$.
   - Si $\Delta < 0$ : accepter ($s \leftarrow s'$).
   - Sinon : accepter avec probabilité $\exp(-\Delta / T)$ (critère de Metropolis).
4. Garder la meilleure solution rencontrée ($s^\star$).

$T$ décroît → l'algorithme passe d'une phase **d'exploration** (accepte beaucoup de dégradations) à une phase **d'exploitation** (devient quasi-glouton). C'est l'analogue exact, from-scratch, de ce que OR-Tools `GUIDED_LOCAL_SEARCH` fait dans le twin Python.


In [7]:
// Cellule 7 — Recuit simule (relocate + 2-opt aleatoire, critere de Metropolis)
public VRPSolution SimulatedAnnealing(VRPInstance inst, VRPSolution initial,
                                       double T0 = 50.0, double TMin = 0.01,
                                       double cooling = 0.9995, int itersPerT = 30,
                                       int seed = 42)
{
    var rng = new Random(seed);
    var current = CloneRoutes(initial.Routes);
    var best = CloneRoutes(initial.Routes);
    double currentCost = SolutionCost(current, inst);
    double bestCost = currentCost;

    double T = T0;
    while (T > TMin)
    {
        for (int k = 0; k < itersPerT; k++)
        {
            var candidate = CloneRoutes(current);
            // Voisin : RELOCATE (deplacer 1 client d'une tournee vers une autre).
            if (candidate.Count >= 2 && rng.NextDouble() < 0.6)
            {
                int fromR = rng.Next(candidate.Count);
                if (candidate[fromR].Count == 0) continue;
                int idx = rng.Next(candidate[fromR].Count);
                int client = candidate[fromR][idx];
                candidate[fromR].RemoveAt(idx);
                int toR = rng.Next(candidate.Count);
                int pos = candidate[toR].Count == 0 ? 0 : rng.Next(candidate[toR].Count + 1);
                // ne garder que si faisable (capacite)
                int newLoad = candidate[toR].Sum(c => inst.Demands[c]) + inst.Demands[client];
                if (newLoad > inst.VehicleCapacity) continue;   // rejete, on retente
                candidate[toR].Insert(pos, client);
            }
            else
            {
                // Voisin : 2-opt sur une tournee aleatoire.
                int r = rng.Next(candidate.Count);
                if (candidate[r].Count >= 3)
                {
                    int i = rng.Next(candidate[r].Count - 1);
                    int j = rng.Next(i + 1, candidate[r].Count);
                    candidate[r].Reverse(i + 1, j - i);
                }
            }
            double candCost = SolutionCost(candidate, inst);
            double delta = candCost - currentCost;
            if (delta < -1e-9 || rng.NextDouble() < Math.Exp(-delta / T))
            {
                current = candidate;
                currentCost = candCost;
                if (candCost < bestCost - 1e-9)
                {
                    best = CloneRoutes(candidate);
                    bestCost = candCost;
                }
            }
        }
        T *= cooling;
    }
    return new VRPSolution(best, inst);
}

private static List<List<int>> CloneRoutes(List<List<int>> routes)
    => routes.Select(r => new List<int>(r)).ToList();

private static double SolutionCost(List<List<int>> routes, VRPInstance inst)
    => routes.Sum(r => RouteDistanceOf(r, inst));

var saSolution = SimulatedAnnealing(instance, twoOptSolution);
display($"Greedy + 2-opt          — distance: {twoOptSolution.TotalDistance():F2}");
display($"+ Recuit simule         — distance: {saSolution.TotalDistance():F2}");
display($"Amelioration SA         — {twoOptSolution.TotalDistance() - saSolution.TotalDistance():F2}");
display($"SA valide: {saSolution.IsValid()}");


Greedy + 2-opt          — distance: 413,69

+ Recuit simule         — distance: 403,45

Amelioration SA         — 10,24

SA valide: True

## 6. Comparaison des méthodes

Tableau récapitulatif sur la **même instance** (15 clients, capacité 60). La qualité augmente avec la sophistication de la méthode, mais aussi le temps de calcul. Le solveur OR-Tools (twin Python) reste la référence SOTA : il combine construction + recherche locale guidée (GLS) avec des heuristiques de coupes avancées.


In [8]:
// Cellule 8 — Tableau comparatif des 4 methodes from-scratch
double dNN = nnSolution.TotalDistance();
double dGreedy = greedySolution.TotalDistance();
double d2opt = twoOptSolution.TotalDistance();
double dSA = saSolution.TotalDistance();
double baseline = dNN;

display("+----------------------+----------+-----------+----------+");
display("| Methode              | Distance | vs NN (%) | Tournees |");
display("+----------------------+----------+-----------+----------+");
display($"| Nearest-Neighbor     | {dNN,8:F2} |  (ref)    | {nnSolution.Routes.Count,8} |");
display($"| Cheapest-Insertion   | {dGreedy,8:F2} | {(dGreedy - baseline) / baseline * 100,+9:F1}% | {greedySolution.Routes.Count,8} |");
display($"| + 2-opt              | {d2opt,8:F2} | {(d2opt - baseline) / baseline * 100,+9:F1}% | {twoOptSolution.Routes.Count,8} |");
display($"| + Recuit simule      | {dSA,8:F2} | {(dSA - baseline) / baseline * 100,+9:F1}% | {saSolution.Routes.Count,8} |");
display("+----------------------+----------+-----------+----------+");

// Coherence mathematique (self-check) : monotonie de l'amelioration.
bool monotone = dSA <= d2opt + 1e-6 && d2opt <= dGreedy + 1e-6;
display(monotone ? "[OK] Mono: SA <= 2-opt <= Greedy (amelioration monotone)."
                 : "[WARN] Non-monotone — verifier les parametres SA.");

// Validite (self-check G.1) : chaque methode produit une solution faisable.
display($"Validite NN={nnSolution.IsValid()}, Greedy={greedySolution.IsValid()}, " +
        $"2-opt={twoOptSolution.IsValid()}, SA={saSolution.IsValid()}");


+----------------------+----------+-----------+----------+

| Methode              | Distance | vs NN (%) | Tournees |

+----------------------+----------+-----------+----------+

| Nearest-Neighbor     |   415,63 |  (ref)    |        6 |

| Cheapest-Insertion   |   413,69 |      -0,5% |        6 |

| + 2-opt              |   413,69 |      -0,5% |        6 |

| + Recuit simule      |   403,45 |      -2,9% |        6 |

+----------------------+----------+-----------+----------+

[OK] Mono: SA <= 2-opt <= Greedy (amelioration monotone).

Validite NN=True, Greedy=True, 2-opt=True, SA=True

## 7. Visualisation ASCII de la meilleure tournée

Le twin Python utilise `matplotlib` pour dessiner les tournées. En C# BCL (0 NuGet), on visualise en **ASCII** : coordonnées (X croissant vers la droite, Y vers le haut) avec le dépôt marqué `D`, chaque tournée par un chiffre distinctif. C'est la convention de visualisation des twins C# Search (cf App-3b/4b).


In [9]:
// Cellule 9 — Carte ASCII de la meilleure solution (recuit simule)
public string DrawAsciiMap(VRPInstance inst, VRPSolution sol, int width = 52, int height = 22)
{
    double minX = Math.Min(inst.Depot.X, inst.Clients.Min(c => c.X));
    double maxX = Math.Max(inst.Depot.X, inst.Clients.Max(c => c.X));
    double minY = Math.Min(inst.Depot.Y, inst.Clients.Min(c => c.Y));
    double maxY = Math.Max(inst.Depot.Y, inst.Clients.Max(c => c.Y));
    var grid = new char[height, width];
    for (int y = 0; y < height; y++) for (int x = 0; x < width; x++) grid[y, x] = '.';

    // Marquer le depot 'D'
    {
        int gx = (int)((inst.Depot.X - minX) / (maxX - minX + 1e-9) * (width - 1));
        int gy = (int)((maxY - inst.Depot.Y) / (maxY - minY + 1e-9) * (height - 1));
        if (gx >= 0 && gx < width && gy >= 0 && gy < height) grid[gy, gx] = 'D';
    }
    // Marquer chaque tournee par son index (1..9, puis '*' au-dela)
    for (int r = 0; r < sol.Routes.Count; r++)
    {
        char mark = r < 9 ? (char)('1' + r) : '*';
        foreach (var c in sol.Routes[r])
        {
            var p = inst.Clients[c];
            int gx = (int)((p.X - minX) / (maxX - minX + 1e-9) * (width - 1));
            int gy = (int)((maxY - p.Y) / (maxY - minY + 1e-9) * (height - 1));
            if (gx >= 0 && gx < width && gy >= 0 && gy < height) grid[gy, gx] = mark;
        }
    }
    var sb = new System.Text.StringBuilder();
    for (int y = 0; y < height; y++)
    {
        for (int x = 0; x < width; x++) sb.Append(grid[y, x]);
        sb.Append('\n');
    }
    return sb.ToString();
}

display("Carte ASCII (D=depot, 1..N=numero de tournee) : meilleur plan SA");
display("X: " + new string('-', 52));
display(DrawAsciiMap(instance, saSolution));
display("Legende: D=depot, chaque chiffre = client servi par la tournee N");


Carte ASCII (D=depot, 1..N=numero de tournee) : meilleur plan SA

X: ----------------------------------------------------

...6..3....6........................................
....................................................
...3................................................
....................................................
....................................................
4.....4.............................................
....................................................
....................................................
...........................................2........
...........................5.......3................
...........................................2........
...........................5...1...5..2.............
....................................................
....................................................
....................................................
....................................................
...............................1....................
....................................................
..............................................

Legende: D=depot, chaque chiffre = client servi par la tournee N

### Détail des tournées optimales (recuit simulé)

Pour chaque tournée : séquence des clients (indice 1-based), charge totale vs capacité, distance de la tournée.


In [10]:
// Cellule 10 — Detail des tournees (recuit simule)
display($"Solution SA — distance totale: {saSolution.TotalDistance():F2}");
display("");
for (int r = 0; r < saSolution.Routes.Count; r++)
{
    var route = saSolution.Routes[r];
    var seq = string.Join(" -> ", route.Select(c => (c + 1).ToString()));
    if (route.Count == 0) seq = "(vide)";
    display($"Tournee {r + 1}: {seq}");
    display($"    charge: {saSolution.RouteLoad(route)}/{instance.VehicleCapacity}, " +
            $"distance: {saSolution.RouteDistance(route):F2}");
}
display("");
display($"Total demande couverte: {saSolution.Routes.SelectMany(r => r).Sum(c => instance.Demands[c])} " +
        $"/ {instance.Demands.Sum()} (attendu: egalite)");


Solution SA — distance totale: 403,45

Tournee 1: 6 -> 7

    charge: 45/60, distance: 40,63

Tournee 2: 3 -> 2 -> 1

    charge: 43/60, distance: 43,57

...

Tournee 5: 8 -> 9 -> 5

    charge: 60/60, distance: 49,63

Tournee 6: 14 -> 10

    charge: 60/60, distance: 94,11

Total demande couverte: 328 / 328 (attendu: egalite)

## 8. Exercices

Trois extensions à compléter (stubs sans erreur — le notebook s'exécute de bout en bout même non complété, cf règle C.1). Chaque exercice précède d'un indice.


### Exercice 1 — Épargne de Clarke-Wright (Clarke-Wright Savings)

L'heuristique **savings** part de $n$ tournées individuelles (dépôt → client $i$ → dépôt) et fusionne itérativement les paires qui maximisent le gain $s_{ij} = d_{0i} + d_{0j} - d_{ij}$. C'est l'heuristique constructive la plus efficace pour le CVRP en pratique.

**Objectif** : implémenter `ClarkeWright(instance)` retournant une `VRPSolution` faisable.

# Indice : calculer la liste triée des savings décroissants, puis fusionner en respectant la capacité et la contrainte « un client n'appartient qu'à une tournée ».


In [11]:
// Cellule 11 — Exercice 1 (a completer) : Clarke-Wright Savings
// TODO etudiant : implementer ClarkeWright(instance) -> VRPSolution
// Etape 1 : partir de n tournees individuelles (1 client chacune).
// Etape 2 : calculer s(i,j) = Dist[0,i+1] + Dist[0,j+1] - Dist[i+1,j+1] pour tout i<j.
// Etape 3 : trier par s decroissant, fusionner les tournees connectees si capacite OK.
public VRPSolution ClarkeWright(VRPInstance inst)
{
    // TODO etudiant
    return null;   // stub : retourner la solution construite
}

// Test (commente tant que l'exercice n'est pas complete) :
// var cwSolution = ClarkeWright(instance);
// display($"Clarke-Wright valide: {cwSolution?.IsValid()}");
// display($"Clarke-Wright distance: {(cwSolution?.TotalDistance() ?? 0):F2}");
display("Exercice 1 (Clarke-Wright) : stub a completer.");


Exercice 1 (Clarke-Wright) : stub a completer.

### Exercice 2 — Mouvement OR-opt

Le **OR-opt** déplace un segment de 1 à 3 clients consécutifs vers une autre position d'une tournée (même tournée ou autre). Combiné au 2-opt, il forme le voisinage standard des métaheuristiques VRP.

**Objectif** : ajouter l'opérateur OR-opt au voisinage du recuit simulé et mesurer l'impact sur la qualité finale.

# Indice : dans `SimulatedAnnealing`, ajouter un 3e type de mouvement (extraire segment length 1-3, réinsérer à une position aléatoire faisable).


In [12]:
// Cellule 12 — Exercice 2 (a completer) : OR-opt dans le recuit simule
// TODO etudiant : etendre SimulatedAnnealing avec un mouvement OR-opt
// Etape 1 : extraire un segment de longueur 1..3 d'une tournee source.
// Etape 2 : le reinserer a une position (autre tournee ou meme) en gardant la capacite.
// Etape 3 : comparer la distance finale avec vs sans OR-opt.
public double SolveSA_WithOrOpt(VRPInstance inst, VRPSolution initial)
{
    // TODO etudiant : reutiliser SimulatedAnnealing en ajoutant OR-opt au voisinage.
    return 0.0;   // stub : retourner la distance finale
}

display("Exercice 2 (OR-opt) : stub a completer. Comparer distance finale avec/sans OR-opt.");


Exercice 2 (OR-opt) : stub a completer. Comparer distance finale avec/sans OR-opt.

### Exercice 3 — VRP avec fenêtres de temps (VRPTW)

Le **VRPTW** ajoute : chaque client $i$ a une fenêtre de temps $[e_i, l_i]$ durant laquelle il doit être servi, et un temps de service $s_i$. Le véhicule quitte le dépôt à $t=0$ et voyage à vitesse unitaire (distance = temps).

**Objectif** : étendre `VRPInstance` en `VRPTWInstance` et valider les fenêtres dans `IsValid()`.

# Indice : calculer le temps d'arrivée à chaque client (somme cumulée des distances + service), vérifier $e_i \le t_i \le l_i$.


In [13]:
// Cellule 13 — Exercice 3 (a completer) : VRPTW
// TODO etudiant : etendre VRPInstance avec time windows et valider
// Etape 1 : ajouter TimeWindows (e_i, l_i) et ServiceTimes par client.
// Etape 2 : dans IsValid(), verifier que l'heure d'arrivee a chaque client est dans [e_i, l_i].
public class VRPTWInstance
{
    public VRPInstance Base;
    // TODO etudiant : ajouter TimeWindows (List<(int earliest, int latest)>) et ServiceTimes.
    // public List<(int Earliest, int Latest)> TimeWindows;
    // public List<int> ServiceTimes;

    public VRPTWInstance(VRPInstance basis) { Base = basis; }

    public bool ValidateTimeWindows(VRPSolution sol)
    {
        // TODO etudiant
        return true;   // stub : a implementer
    }
}

display("Exercice 3 (VRPTW) : stub a completer.");


Exercice 3 (VRPTW) : stub a completer.

## 9. Résumé

Ce twin C# a déroulé **toutes les briques** d'un solveur VRP, from-scratch (BCL .NET 9, 0 NuGet) :

1. **Modélisation** : `VRPInstance` (dépôt, clients, demandes, matrice euclidienne) + `VRPSolution` (tournées, coût, validité capacité).
2. **Heuristiques constructives** : plus proche voisin (myope, $O(n^2)$), insertion au moindre coût (meilleure vision globale).
3. **Recherche locale 2-opt** : élimine les croisements d'arcs, atteint un optimum local.
4. **Recuit simulé** : échappe aux optima locaux via le critère de Metropolis, refroidissement géométrique.

### Leçons clés

- **Monotonie de l'amélioration** : NN ≥ Cheapest-Insertion ≥ +2-opt ≥ +SA (vérifié par self-check). Chaque couche apporte un gain décroissant mais réel.
- **Myopie vs vision globale** : NN est rapide mais entasse ; l'insertion au moindre coût et le 2-opt corrigent ses défauts structurels.
- **Le recuit simulé lève le verrou de l'optimum local** : c'est l'opérateur qui rapproche le plus le from-scratch du solveur SOTA OR-Tools (GLS).

### Perspectives

- **Clarke-Wright savings** (exercice 1) : heuristique constructive de référence, souvent meilleure que l'insertion.
- **Mouvements OR-opt** (exercice 2) : voisinage plus riche pour le recuit.
- **VRPTW** (exercice 3) : extension avec fenêtres de temps (contraintes supplémentaires).

### Référence SOTA

Le twin Python [App-17-VRP-Logistics](App-17-VRP-Logistics.ipynb) résout la même instance avec **OR-Tools `RoutingModel`** + Guided Local Search. OR-Tools orchestre en interne (et de manière optimisée en C++) les briques que ce twin explicite : construction `PATH_CHEAPEST_ARC`, recherche locale, métaheuristique GLS. En production : OR-Tools. En pédagogie : comprendre chaque brique, from-scratch.

---

**Marathon #4956** — twin C# .NET ⇄ Python. Voir #4956, #3801. Revue souhaitée : ai-01, po-2024 (Search/.NET owner), po-2025.
